# Cài đặt

In [ ]:
import os

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive/')


# Hàm đếm số lượng file trong các thư mục
def count_files_in_folders(folders):
    for folder in folders:
        file_count = 0
        for dirpath, dirnames, filenames in os.walk(folder):
            file_count += len(filenames)
        print(f"Tổng số file trong thư mục '{folder}': {file_count}")

Mounted at /content/drive/


In [ ]:
%pip install --quiet PyMuPDF nltk pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 43.7 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="MuPDF error")

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

ROOT_FOLDER = '/content/drive/MyDrive/PETTIES/DATA BÁO CÁO - nguồn để tính EW_ESGUI'

def count_for_path(path):
    errs = []
    def _onerror(e):
        errs.append(str(e))
    total = 0
    for _, _, files in os.walk(path, onerror=_onerror):
        total += len(files)
    return total, errs

def count_files_and_report_unreadable(root_folder):
    if not os.path.isdir(root_folder):
        print(f"Không tìm thấy thư mục: {root_folder}")
        return

    unreadable = []   # [(subfolder_name, error_msg)]
    results = []      # [(subfolder_name, count)]

    for entry in sorted(os.scandir(root_folder), key=lambda x: x.name):
        if entry.is_dir(follow_symlinks=False):
            count, errs = count_for_path(entry.path)
            if errs:
                unreadable.append((entry.name, "; ".join(set(errs))))
            results.append((entry.name, count))

    print(f"Tổng số file trong '{root_folder}': {sum(c for _, c in results)}")
    for name, c in results:
        print(f"{name}: {c} files")

    if unreadable:
        print("\nCác folder không đọc được (có lỗi):")
        for name, msg in unreadable:
            print(f"- {name}: {msg}")
    else:
        print("\nKhông phát hiện folder nào lỗi/không đọc được.")

count_files_and_report_unreadable(ROOT_FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tổng số file trong '/content/drive/MyDrive/PETTIES/DATA BÁO CÁO - nguồn để tính EW_ESGUI': 7831
 SAUDI ARABIA: 165 files
CHINA: 377 files
GEORGIA: 158 files
INDIA: 466 files
INDONESIA: 139 files
IRAN: 200 files
IRAQ: 108 files
ISRAEL: 141 files
JAPAN: 535 files
JORDAN: 196 files
KOREA: 357 files
MALAYSIA: 146 files
PHILIPPINES: 136 files
QATAR: 99 files
SINGAPORE: 327 files
TAJIKISTAN: 205 files
THAILAND: 225 files
VIETNAM : 346 files
nháp: 3505 files

Không phát hiện folder nào lỗi/không đọc được.


# CHINA

In [ ]:
##TRUNG QUỐC
##kw cũ
#zcore
import os, re
import fitz  # PyMuPDF
import nltk
import pandas as pd
import numpy as np
from scipy.stats import norm
from nltk.tokenize import word_tokenize
from collections import Counter

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# NLTK setup
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# ====== 1) ĐỌC PDF & TIỀN XỬ LÝ ======
def read_pdf_with_fitz(path):
    doc = fitz.open(path)
    text = []
    for page in doc:
        text.append(page.get_text("text"))
    return "\n".join(text)

def preprocess(raw):
    clean = raw.lower()
    clean = clean.replace("co₂", "co2").replace("so₂", "so2")
    clean = re.sub(r'[^a-z\s]', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    tokens = word_tokenize(clean)
    return clean, tokens

# ====== 2) TỪ KHÓA/CỤM TỪ ======
E_keywords = [
    "acid","agriculture","biodiversity","carbon","clean","co2","conservation","deforestation",
    "depletion","drought","ecofriendly","ecology","emission","energy","erosion","flood","footprint",
    "fossil","geothermal","green","greenhouse","hydro","landfill","methane","microplastics","monoxide",
    "natural","noise","oil","organic","overconsumption","overfishing","overpopulation","ozone",
    "pesticide","plastic","poison","pollution","radiation","recycling","renewable","resources",
    "so2","smog","soil","solar","storm","sustainability","toxicity","trash","waste","wastewater","wind"
]
E_phrases = ["carbon footprint","carbon emission","renewable energy","wind energy","greenhouse gas"]

S_keywords = [
    "agility","career","charitableness","collaboration","communication","conscience","corruption","crime",
    "culture","dialogue","discrimination","donation","donations","elderly","empathy","employment",
    "entrepreneurship","equal","female","flexibility","freedom","gender","harassment","harmony","honesty",
    "humanity","innovation","integration","mentorship","morality","participation","profession","respect",
    "responsibility","health","community","inclusion","wellbeing","safety","privacy","labor"
]
S_phrases = ["gender equality","labor rights","human rights","data privacy","workplace safety","community development"]

G_keywords = [
    "accountability","adaptability","agility","audit","autonomy","board","coherence","collaboration",
    "communication","compensation","compliance","consistency","corporate","culture","corruption","data",
    "protection","decisions","diligence","diversity","efficiency","ethics","evaluation","fairness",
    "leadership","legitimacy","meritocracy","objectivity","oversight","performance","principle",
    "prudence","quality","reciprocity","relations","reliability","resilience","responsibility","risk",
    "stakeholders","stewardship","strategy","sustainability","transparency"
]
G_phrases = ["board independence","executive compensation","shareholder rights","stakeholder engagement",
             "anti corruption","anti-corruption","data protection"]
UI_keywords = ["uncertain","uncertainty","uncertainties","risk","risks","volatile","volatility","instability","ambiguity","fluctuation","doubt"]

# ====== 3) HÀM ĐẾM ======
def count_words(tokens, vocab):
    vocab = set(w.lower() for w in vocab)
    c = Counter(tokens)
    return sum(c[w] for w in vocab)

def count_phrases(clean_text, phrases):
    total = 0
    for p in phrases:
        pat = r'\b' + re.sub(r'\s+', r'\\s+', re.escape(p.lower())) + r'\b'
        total += len(re.findall(pat, clean_text))
    return total

# ====== 4) SAFE Z + CDF chuyển về 0–100 ======
def safe_z(x, mu, sigma):
    return 0 if sigma == 0 or np.isnan(sigma) else (x - mu) / sigma

def z_to_score(z):
    """Convert Z to 0–100 scale using Normal CDF"""
    return round(norm.cdf(z), 4)

# ====== 5) CẤU HÌNH ======
BASE_DIR = '/content/drive/MyDrive/PETTIES/DATA BÁO CÁO - nguồn để tính EW_ESGUI/CHINA'
YEARS = [str(y) for y in range(2002, 2026)]

yearly_data = {}
years_with_no_pdf = []
unreadable_items = []

# ====== 6) ĐỌC PDF THEO NĂM ======
for year in YEARS:
    year_dir = os.path.join(BASE_DIR, year)
    if not os.path.isdir(year_dir):
        years_with_no_pdf.append((year, "no_folder"))
        continue

    pdfs = []
    for dp, dn, fn in os.walk(year_dir):
        for f in fn:
            if f.lower().endswith(".pdf"):
                pdfs.append(os.path.join(dp, f))
    if not pdfs:
        years_with_no_pdf.append((year, "no_pdf"))
        continue

    yearly_data[year] = {"E":0.0, "S":0.0, "G":0.0, "UI":0.0, "total_tokens":0, "count":0}

    for fpath in pdfs:
        try:
            raw = read_pdf_with_fitz(fpath)
        except Exception as e:
            unreadable_items.append((year, fpath, str(e)))
            continue

        clean, tokens = preprocess(raw)
        total_tokens = len(tokens)
        if total_tokens == 0:
            continue

        E_cnt  = count_words(tokens, E_keywords) + count_phrases(clean, E_phrases)
        S_cnt  = count_words(tokens, S_keywords) + count_phrases(clean, S_phrases)
        G_cnt  = count_words(tokens, G_keywords) + count_phrases(clean, G_phrases)
        UI_cnt = count_words(tokens, UI_keywords)

        yd = yearly_data[year]
        yd["E"] += E_cnt
        yd["S"] += S_cnt
        yd["G"] += G_cnt
        yd["UI"] += UI_cnt
        yd["total_tokens"] += total_tokens
        yd["count"] += 1

# ====== 7) TÍNH AVERAGE ======
stats_rows = []
for year in YEARS:
    if year not in yearly_data:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue
    d = yearly_data[year]
    n, tok = d["count"], d["total_tokens"]
    if n == 0 or tok == 0:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue

    E_avg  = d["E"]/tok
    S_avg  = d["S"]/tok
    G_avg  = d["G"]/tok
    UI_avg = d["UI"]/tok

    stats_rows.append({"Year": year, "N_Files": n,
                       "E_avg": E_avg, "S_avg": S_avg, "G_avg": G_avg, "UI_avg": UI_avg})

df_stats = pd.DataFrame(stats_rows)

mask = df_stats["N_Files"] > 0
E_mu,  E_sd  = df_stats.loc[mask, "E_avg"].mean(), df_stats.loc[mask, "E_avg"].std(ddof=0)
S_mu,  S_sd  = df_stats.loc[mask, "S_avg"].mean(), df_stats.loc[mask, "S_avg"].std(ddof=0)
G_mu,  G_sd  = df_stats.loc[mask, "G_avg"].mean(), df_stats.loc[mask, "G_avg"].std(ddof=0)
UI_mu, UI_sd = df_stats.loc[mask, "UI_avg"].mean(), df_stats.loc[mask, "UI_avg"].std(ddof=0)

# ====== 😎 CHUẨN HÓA VỀ 0–100 ======
rows = []
for _, r in df_stats.iterrows():
    year = r["Year"]
    n    = r["N_Files"]
    if n == 0 or pd.isna(r["E_avg"]):
        rows.append({
            "Year": year, "N_Files": 0,
            "E_score_0_100": None, "S_score_0_100": None, "G_score_0_100": None, "UI_score_0_100": None,
            "ESG_subindex_0_100": None, "ESGUI_0_100": None
        })
        continue

    E_z, S_z, G_z, UI_z = (
        safe_z(r["E_avg"], E_mu, E_sd),
        safe_z(r["S_avg"], S_mu, S_sd),
        safe_z(r["G_avg"], G_mu, G_sd),
        safe_z(r["UI_avg"], UI_mu, UI_sd),
    )

    # convert Z to 0–100
    E_sc, S_sc, G_sc, UI_sc = (
        z_to_score(E_z),
        z_to_score(S_z),
        z_to_score(G_z),
        z_to_score(UI_z)
    )

    ESG_sub = round((E_sc + S_sc + G_sc)/3, 4)
    ESGUI   = round(0.5*ESG_sub + 0.5*UI_sc, 4)

    rows.append({
        "Year": year, "N_Files": int(n),
        "E_score_0_100": E_sc,
        "S_score_0_100": S_sc,
        "G_score_0_100": G_sc,
        "UI_score_0_100": UI_sc,
        "ESG_subindex_0_100": ESG_sub,
        "ESGUI_0_100": ESGUI
    })

df_yearly = pd.DataFrame(rows).sort_values("Year")
print("=== YEARLY SUMMARY (Z-score → 0–100) ===")
print(df_yearly.to_string(index=False))

# ====== 9) LOG ======
if years_with_no_pdf:
    print("\n-- Years missing data --")
    for y, typ in years_with_no_pdf:
        msg = "folder not found" if typ == "no_folder" else "no PDF found"
        print(f"{y}: {msg}")

if unreadable_items:
    print("\n-- Unreadable files --")
    for y, p, e in unreadable_items[:50]:
        print(f"[{y}] {p} -> {e}")
    if len(unreadable_items) > 50:
        print(f"... and {len(unreadable_items)-50} more")

print("\n--- Reference statistics ---")
print(f"E_mu={E_mu:.8f}, E_sd={E_sd:.8f}")
print(f"S_mu={S_mu:.8f}, S_sd={S_sd:.8f}")
print(f"G_mu={G_mu:.8f}, G_sd={G_sd:.8f}")
print(f"UI_mu={UI_mu:.8f}, UI_sd={UI_sd:.8f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== YEARLY SUMMARY (Z-score → 0–100) ===
Year  N_Files  E_score_0_100  S_score_0_100  G_score_0_100  UI_score_0_100  ESG_subindex_0_100  ESGUI_0_100
2002       14         0.1260         0.4903         0.2717          0.2427              0.2960       0.2694
2003       15         0.2416         0.0485         0.3650          0.2037              0.2184       0.2110
2004       15         0.2401         0.0674         0.2542          0.2747              0.1872       0.2310
2005       15         0.3730         0.2204         0.2639          0.2590              0.2858       0.2724
2006       13         0.2330         0.8873         0.2228          0.1655              0.4477       0.3066
2007       13         0.3635         0.9700         0.3369          0.2255              0.5568       0.3912
2008       15         0.2170         0.6224         0.1305          0.2234

In [ ]:
import numpy as np
import pandas as pd

# ---- 1) Chuẩn bị dữ liệu ----
china = df = df_yearly.copy()

# ---- 2) Lấy 3 cột ESG Z→CDF→0–100 ----
ESG_cols = ["E_score_0_100", "S_score_0_100", "G_score_0_100"]

# ---- 3) DÙNG TRỰC TIẾP dữ liệu Z-score-0–100 cho entropy ----
X = df[ESG_cols].astype(float)

# Thêm 0.001 để tránh log(0) – GIỮ nguyên đúng công thức entropy
X_safe = X + 0.001

# ---- 4) p_ij = x_ij / sum(x_ij) (theo cột) ----
P = X_safe.div(X_safe.sum(axis=0), axis=1)

# ---- 5) Entropy ----
n = len(df)   # số năm
e = -(P * np.log(P)).sum(axis=0) / np.log(n)

# ---- 6) Divergence + Weights ----
d = 1 - e
w = d / d.sum()

print("Entropy weights (E, S, G):")
print(w.to_dict())

# ---- 7) Weighted ESG ----
df["E_weighted"] = df["E_score_0_100"] * w["E_score_0_100"]
df["S_weighted"] = df["S_score_0_100"] * w["S_score_0_100"]
df["G_weighted"] = df["G_score_0_100"] * w["G_score_0_100"]

df["ESG_sub_entropy"] = df["E_weighted"] + df["S_weighted"] + df["G_weighted"]

# ---- 😎 ESGUI mới ----
df["ESGUI_new"] = 0.5 * df["ESG_sub_entropy"] + 0.5 * df["UI_score_0_100"]

# ---- 9) Xuất bảng ----
pd.set_option('display.max_columns', None)

print(df[[
    "Year",
    "E_score_0_100", "S_score_0_100", "G_score_0_100",
    "E_weighted", "S_weighted", "G_weighted",
    "ESG_sub_entropy", "UI_score_0_100", "ESGUI_new"
]])

Entropy weights (E, S, G):
{'E_score_0_100': 0.3001626064728123, 'S_score_0_100': 0.3875992875749844, 'G_score_0_100': 0.31223810595220325}
    Year  E_score_0_100  S_score_0_100  G_score_0_100  E_weighted  S_weighted  \
0   2002         0.1260         0.4903         0.2717    0.037820    0.190040   
1   2003         0.2416         0.0485         0.3650    0.072519    0.018799   
2   2004         0.2401         0.0674         0.2542    0.072069    0.026124   
3   2005         0.3730         0.2204         0.2639    0.111961    0.085427   
4   2006         0.2330         0.8873         0.2228    0.069938    0.343917   
5   2007         0.3635         0.9700         0.3369    0.109109    0.375971   
6   2008         0.2170         0.6224         0.1305    0.065135    0.241242   
7   2009         0.1988         0.6288         0.1548    0.059672    0.243722   
8   2010         0.2543         0.7865         0.2222    0.076331    0.304847   
9   2011         0.3726         0.7812         0.2

In [ ]:
import os

# 1. Định nghĩa đường dẫn thư mục và file output
output_dir = "/content/drive/MyDrive/PETTIES/OUTPUT"
output_file = os.path.join(output_dir, "Chia_ESG_Entropy_Results.xlsx") # Hoặc đổi đuôi thành .csv nếu muốn

# 2. Tự động kiểm tra và tạo folder nếu chưa tồn tại
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"🎉 Đã tạo thành công thư mục mới tại: {output_dir}")
else:
    print(f"📁 Thư mục đã tồn tại: {output_dir}")

# 3. Lọc các cột cần thiết để xuất (hoặc xuất hết cả df bằng cách bỏ filter)
cols_to_export = [
    "Year",
    "E_score_0_100", "S_score_0_100", "G_score_0_100",
    "E_weighted", "S_weighted", "G_weighted",
    "ESG_sub_entropy", "UI_score_0_100", "ESGUI_new"
]
df_output = df[cols_to_export]

# 4. Tiến hành xuất file
try:
    # Nếu chọn xuất Excel
    df_output.to_excel(output_file, index=False, engine='openpyxl')
    print(f"💾 Đã lưu dữ liệu thành công vào file Excel: {output_file}")

    # Kèm theo file CSV dự phòng (nếu muốn, có thể comment lại)
    # csv_file = output_file.replace(".xlsx", ".csv")
    # df_output.to_csv(csv_file, index=False, encoding='utf-8-sig')
    # print(f"💾 Đã lưu thêm bản dự phòng CSV: {csv_file}")

except Exception as e:
    print(f"❌ Có lỗi xảy ra khi lưu file: {e}")
    print("💡 Lưu ý: Hãy chắc chắn rằng bạn đã mount Google Drive thành công trước đó bằng lệnh: drive.mount('/content/drive')")

📁 Thư mục đã tồn tại: /content/drive/MyDrive/PETTIES/OUTPUT
💾 Đã lưu dữ liệu thành công vào file Excel: /content/drive/MyDrive/PETTIES/OUTPUT/Chia_ESG_Entropy_Results.xlsx


# INDIA

In [ ]:
##TRUNG QUỐC
##kw cũ
#zcore
import os, re
import fitz  # PyMuPDF
import nltk
import pandas as pd
import numpy as np
from scipy.stats import norm
from nltk.tokenize import word_tokenize
from collections import Counter

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# NLTK setup
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# ====== 1) ĐỌC PDF & TIỀN XỬ LÝ ======
def read_pdf_with_fitz(path):
    doc = fitz.open(path)
    text = []
    for page in doc:
        text.append(page.get_text("text"))
    return "\n".join(text)

def preprocess(raw):
    clean = raw.lower()
    clean = clean.replace("co₂", "co2").replace("so₂", "so2")
    clean = re.sub(r'[^a-z\s]', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    tokens = word_tokenize(clean)
    return clean, tokens

# ====== 2) TỪ KHÓA/CỤM TỪ ======
E_keywords = [
    "acid","agriculture","biodiversity","carbon","clean","co2","conservation","deforestation",
    "depletion","drought","ecofriendly","ecology","emission","energy","erosion","flood","footprint",
    "fossil","geothermal","green","greenhouse","hydro","landfill","methane","microplastics","monoxide",
    "natural","noise","oil","organic","overconsumption","overfishing","overpopulation","ozone",
    "pesticide","plastic","poison","pollution","radiation","recycling","renewable","resources",
    "so2","smog","soil","solar","storm","sustainability","toxicity","trash","waste","wastewater","wind"
]
E_phrases = ["carbon footprint","carbon emission","renewable energy","wind energy","greenhouse gas"]

S_keywords = [
    "agility","career","charitableness","collaboration","communication","conscience","corruption","crime",
    "culture","dialogue","discrimination","donation","donations","elderly","empathy","employment",
    "entrepreneurship","equal","female","flexibility","freedom","gender","harassment","harmony","honesty",
    "humanity","innovation","integration","mentorship","morality","participation","profession","respect",
    "responsibility","health","community","inclusion","wellbeing","safety","privacy","labor"
]
S_phrases = ["gender equality","labor rights","human rights","data privacy","workplace safety","community development"]

G_keywords = [
    "accountability","adaptability","agility","audit","autonomy","board","coherence","collaboration",
    "communication","compensation","compliance","consistency","corporate","culture","corruption","data",
    "protection","decisions","diligence","diversity","efficiency","ethics","evaluation","fairness",
    "leadership","legitimacy","meritocracy","objectivity","oversight","performance","principle",
    "prudence","quality","reciprocity","relations","reliability","resilience","responsibility","risk",
    "stakeholders","stewardship","strategy","sustainability","transparency"
]
G_phrases = ["board independence","executive compensation","shareholder rights","stakeholder engagement",
             "anti corruption","anti-corruption","data protection"]
UI_keywords = ["uncertain","uncertainty","uncertainties","risk","risks","volatile","volatility","instability","ambiguity","fluctuation","doubt"]

# ====== 3) HÀM ĐẾM ======
def count_words(tokens, vocab):
    vocab = set(w.lower() for w in vocab)
    c = Counter(tokens)
    return sum(c[w] for w in vocab)

def count_phrases(clean_text, phrases):
    total = 0
    for p in phrases:
        pat = r'\b' + re.sub(r'\s+', r'\\s+', re.escape(p.lower())) + r'\b'
        total += len(re.findall(pat, clean_text))
    return total

# ====== 4) SAFE Z + CDF chuyển về 0–100 ======
def safe_z(x, mu, sigma):
    return 0 if sigma == 0 or np.isnan(sigma) else (x - mu) / sigma

def z_to_score(z):
    """Convert Z to 0–100 scale using Normal CDF"""
    return round(norm.cdf(z), 4)

# ====== 5) CẤU HÌNH ======
BASE_DIR = '/content/drive/MyDrive/PETTIES/DATA BÁO CÁO - nguồn để tính EW_ESGUI/Ấn'
YEARS = [str(y) for y in range(2002, 2025)]

yearly_data = {}
years_with_no_pdf = []
unreadable_items = []

# ====== 6) ĐỌC PDF THEO NĂM ======
for year in YEARS:
    year_dir = os.path.join(BASE_DIR, year)
    if not os.path.isdir(year_dir):
        years_with_no_pdf.append((year, "no_folder"))
        continue

    pdfs = []
    for dp, dn, fn in os.walk(year_dir):
        for f in fn:
            if f.lower().endswith(".pdf"):
                pdfs.append(os.path.join(dp, f))
    if not pdfs:
        years_with_no_pdf.append((year, "no_pdf"))
        continue

    yearly_data[year] = {"E":0.0, "S":0.0, "G":0.0, "UI":0.0, "total_tokens":0, "count":0}

    for fpath in pdfs:
        try:
            raw = read_pdf_with_fitz(fpath)
        except Exception as e:
            unreadable_items.append((year, fpath, str(e)))
            continue

        clean, tokens = preprocess(raw)
        total_tokens = len(tokens)
        if total_tokens == 0:
            continue

        E_cnt  = count_words(tokens, E_keywords) + count_phrases(clean, E_phrases)
        S_cnt  = count_words(tokens, S_keywords) + count_phrases(clean, S_phrases)
        G_cnt  = count_words(tokens, G_keywords) + count_phrases(clean, G_phrases)
        UI_cnt = count_words(tokens, UI_keywords)

        yd = yearly_data[year]
        yd["E"] += E_cnt
        yd["S"] += S_cnt
        yd["G"] += G_cnt
        yd["UI"] += UI_cnt
        yd["total_tokens"] += total_tokens
        yd["count"] += 1

# ====== 7) TÍNH AVERAGE ======
stats_rows = []
for year in YEARS:
    if year not in yearly_data:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue
    d = yearly_data[year]
    n, tok = d["count"], d["total_tokens"]
    if n == 0 or tok == 0:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue

    E_avg  = d["E"]/tok
    S_avg  = d["S"]/tok
    G_avg  = d["G"]/tok
    UI_avg = d["UI"]/tok

    stats_rows.append({"Year": year, "N_Files": n,
                       "E_avg": E_avg, "S_avg": S_avg, "G_avg": G_avg, "UI_avg": UI_avg})

df_stats = pd.DataFrame(stats_rows)

mask = df_stats["N_Files"] > 0
E_mu,  E_sd  = df_stats.loc[mask, "E_avg"].mean(), df_stats.loc[mask, "E_avg"].std(ddof=0)
S_mu,  S_sd  = df_stats.loc[mask, "S_avg"].mean(), df_stats.loc[mask, "S_avg"].std(ddof=0)
G_mu,  G_sd  = df_stats.loc[mask, "G_avg"].mean(), df_stats.loc[mask, "G_avg"].std(ddof=0)
UI_mu, UI_sd = df_stats.loc[mask, "UI_avg"].mean(), df_stats.loc[mask, "UI_avg"].std(ddof=0)

# ====== 😎 CHUẨN HÓA VỀ 0–100 ======
rows = []
for _, r in df_stats.iterrows():
    year = r["Year"]
    n    = r["N_Files"]
    if n == 0 or pd.isna(r["E_avg"]):
        rows.append({
            "Year": year, "N_Files": 0,
            "E_score_0_100": None, "S_score_0_100": None, "G_score_0_100": None, "UI_score_0_100": None,
            "ESG_subindex_0_100": None, "ESGUI_0_100": None
        })
        continue

    E_z, S_z, G_z, UI_z = (
        safe_z(r["E_avg"], E_mu, E_sd),
        safe_z(r["S_avg"], S_mu, S_sd),
        safe_z(r["G_avg"], G_mu, G_sd),
        safe_z(r["UI_avg"], UI_mu, UI_sd),
    )

    # convert Z to 0–100
    E_sc, S_sc, G_sc, UI_sc = (
        z_to_score(E_z),
        z_to_score(S_z),
        z_to_score(G_z),
        z_to_score(UI_z)
    )

    ESG_sub = round((E_sc + S_sc + G_sc)/3, 4)
    ESGUI   = round(0.5*ESG_sub + 0.5*UI_sc, 4)

    rows.append({
        "Year": year, "N_Files": int(n),
        "E_score_0_100": E_sc,
        "S_score_0_100": S_sc,
        "G_score_0_100": G_sc,
        "UI_score_0_100": UI_sc,
        "ESG_subindex_0_100": ESG_sub,
        "ESGUI_0_100": ESGUI
    })

df_yearly = pd.DataFrame(rows).sort_values("Year")
print("=== YEARLY SUMMARY (Z-score → 0–100) ===")
print(df_yearly.to_string(index=False))

# ====== 9) LOG ======
if years_with_no_pdf:
    print("\n-- Years missing data --")
    for y, typ in years_with_no_pdf:
        msg = "folder not found" if typ == "no_folder" else "no PDF found"
        print(f"{y}: {msg}")

if unreadable_items:
    print("\n-- Unreadable files --")
    for y, p, e in unreadable_items[:50]:
        print(f"[{y}] {p} -> {e}")
    if len(unreadable_items) > 50:
        print(f"... and {len(unreadable_items)-50} more")

print("\n--- Reference statistics ---")
print(f"E_mu={E_mu:.8f}, E_sd={E_sd:.8f}")
print(f"S_mu={S_mu:.8f}, S_sd={S_sd:.8f}")
print(f"G_mu={G_mu:.8f}, G_sd={G_sd:.8f}")
print(f"UI_mu={UI_mu:.8f}, UI_sd={UI_sd:.8f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: library error: FT_New_Memory_Face(QBZAMN+ArnoPro-Smbd): unknown file format

=== YEARLY SUMMARY (Z-score → 0–100) ===
Year  N_Files  E_score_0_100  S_score_0_100  G_score_0_100  UI_score_0_100  ESG_subindex_0_100  ESGUI_0_100
2002       16         0.9994         0.0397         0.1872          0.2635              0.4088       0.3362
2003       23         0.1046         0.0167         0.0281          0.3725              0.0498       0.2112
2004       26         0.9225         0.3106         0.2821          0.2456              0.5051       0.3754
2005       25         0.2972         0.8634         0.1135          0.1794              0.4247       0.3021
2006       23         0.1496         0.8139         0.1748          0.1627              0.3794       0.2710
2007       23         0.4707       

In [ ]:
import numpy as np
import pandas as pd

# ---- 1) Chuẩn bị dữ liệu ----
india = df = df_yearly.copy()

# ---- 2) Lấy 3 cột ESG Z→CDF→0–100 ----
ESG_cols = ["E_score_0_100", "S_score_0_100", "G_score_0_100"]

# ---- 3) DÙNG TRỰC TIẾP dữ liệu Z-score-0–100 cho entropy ----
X = df[ESG_cols].astype(float)

# Thêm 0.001 để tránh log(0) – GIỮ nguyên đúng công thức entropy
X_safe = X + 0.001

# ---- 4) p_ij = x_ij / sum(x_ij) (theo cột) ----
P = X_safe.div(X_safe.sum(axis=0), axis=1)

# ---- 5) Entropy ----
n = len(df)   # số năm
e = -(P * np.log(P)).sum(axis=0) / np.log(n)

# ---- 6) Divergence + Weights ----
d = 1 - e
w = d / d.sum()

print("Entropy weights (E, S, G):")
print(w.to_dict())

# ---- 7) Weighted ESG ----
df["E_weighted"] = df["E_score_0_100"] * w["E_score_0_100"]
df["S_weighted"] = df["S_score_0_100"] * w["S_score_0_100"]
df["G_weighted"] = df["G_score_0_100"] * w["G_score_0_100"]

df["ESG_sub_entropy"] = df["E_weighted"] + df["S_weighted"] + df["G_weighted"]

# ---- 😎 ESGUI mới ----
df["ESGUI_new"] = 0.5 * df["ESG_sub_entropy"] + 0.5 * df["UI_score_0_100"]

# ---- 9) Xuất bảng ----
pd.set_option('display.max_columns', None)

print(df[[
    "Year",
    "E_score_0_100", "S_score_0_100", "G_score_0_100",
    "E_weighted", "S_weighted", "G_weighted",
    "ESG_sub_entropy", "UI_score_0_100", "ESGUI_new"
]])

Entropy weights (E, S, G):
{'E_score_0_100': 0.3006766273034514, 'S_score_0_100': 0.3454412694601689, 'G_score_0_100': 0.35388210323637964}
    Year  E_score_0_100  S_score_0_100  G_score_0_100  E_weighted  S_weighted  \
0   2002         0.9994         0.0397         0.1872    0.300496    0.013714   
1   2003         0.1046         0.0167         0.0281    0.031451    0.005769   
2   2004         0.9225         0.3106         0.2821    0.277374    0.107294   
3   2005         0.2972         0.8634         0.1135    0.089361    0.298254   
4   2006         0.1496         0.8139         0.1748    0.044981    0.281155   
5   2007         0.4707         0.5242         0.1591    0.141528    0.181080   
6   2008         0.7819         0.1823         0.3681    0.235099    0.062974   
7   2009         0.5878         0.1390         0.3260    0.176738    0.048016   
8   2010         0.6123         0.3952         0.3840    0.184104    0.136518   
9   2011         0.6287         0.7639         0.3

# JAPAN

In [ ]:
##TRUNG QUỐC
##kw cũ
#zcore
import os, re
import fitz  # PyMuPDF
import nltk
import pandas as pd
import numpy as np
from scipy.stats import norm
from nltk.tokenize import word_tokenize
from collections import Counter

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# NLTK setup
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# ====== 1) ĐỌC PDF & TIỀN XỬ LÝ ======
def read_pdf_with_fitz(path):
    doc = fitz.open(path)
    text = []
    for page in doc:
        text.append(page.get_text("text"))
    return "\n".join(text)

def preprocess(raw):
    clean = raw.lower()
    clean = clean.replace("co₂", "co2").replace("so₂", "so2")
    clean = re.sub(r'[^a-z\s]', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    tokens = word_tokenize(clean)
    return clean, tokens

# ====== 2) TỪ KHÓA/CỤM TỪ ======
E_keywords = [
    "acid","agriculture","biodiversity","carbon","clean","co2","conservation","deforestation",
    "depletion","drought","ecofriendly","ecology","emission","energy","erosion","flood","footprint",
    "fossil","geothermal","green","greenhouse","hydro","landfill","methane","microplastics","monoxide",
    "natural","noise","oil","organic","overconsumption","overfishing","overpopulation","ozone",
    "pesticide","plastic","poison","pollution","radiation","recycling","renewable","resources",
    "so2","smog","soil","solar","storm","sustainability","toxicity","trash","waste","wastewater","wind"
]
E_phrases = ["carbon footprint","carbon emission","renewable energy","wind energy","greenhouse gas"]

S_keywords = [
    "agility","career","charitableness","collaboration","communication","conscience","corruption","crime",
    "culture","dialogue","discrimination","donation","donations","elderly","empathy","employment",
    "entrepreneurship","equal","female","flexibility","freedom","gender","harassment","harmony","honesty",
    "humanity","innovation","integration","mentorship","morality","participation","profession","respect",
    "responsibility","health","community","inclusion","wellbeing","safety","privacy","labor"
]
S_phrases = ["gender equality","labor rights","human rights","data privacy","workplace safety","community development"]

G_keywords = [
    "accountability","adaptability","agility","audit","autonomy","board","coherence","collaboration",
    "communication","compensation","compliance","consistency","corporate","culture","corruption","data",
    "protection","decisions","diligence","diversity","efficiency","ethics","evaluation","fairness",
    "leadership","legitimacy","meritocracy","objectivity","oversight","performance","principle",
    "prudence","quality","reciprocity","relations","reliability","resilience","responsibility","risk",
    "stakeholders","stewardship","strategy","sustainability","transparency"
]
G_phrases = ["board independence","executive compensation","shareholder rights","stakeholder engagement",
             "anti corruption","anti-corruption","data protection"]
UI_keywords = ["uncertain","uncertainty","uncertainties","risk","risks","volatile","volatility","instability","ambiguity","fluctuation","doubt"]

# ====== 3) HÀM ĐẾM ======
def count_words(tokens, vocab):
    vocab = set(w.lower() for w in vocab)
    c = Counter(tokens)
    return sum(c[w] for w in vocab)

def count_phrases(clean_text, phrases):
    total = 0
    for p in phrases:
        pat = r'\b' + re.sub(r'\s+', r'\\s+', re.escape(p.lower())) + r'\b'
        total += len(re.findall(pat, clean_text))
    return total

# ====== 4) SAFE Z + CDF chuyển về 0–100 ======
def safe_z(x, mu, sigma):
    return 0 if sigma == 0 or np.isnan(sigma) else (x - mu) / sigma

def z_to_score(z):
    """Convert Z to 0–100 scale using Normal CDF"""
    return round(norm.cdf(z), 4)

# ====== 5) CẤU HÌNH ======
BASE_DIR = '/content/drive/MyDrive/PETTIES/DATA BÁO CÁO - nguồn để tính EW_ESGUI/JAPAN'
YEARS = [str(y) for y in range(2002, 2025)]

yearly_data = {}
years_with_no_pdf = []
unreadable_items = []

# ====== 6) ĐỌC PDF THEO NĂM ======
for year in YEARS:
    year_dir = os.path.join(BASE_DIR, year)
    if not os.path.isdir(year_dir):
        years_with_no_pdf.append((year, "no_folder"))
        continue

    pdfs = []
    for dp, dn, fn in os.walk(year_dir):
        for f in fn:
            if f.lower().endswith(".pdf"):
                pdfs.append(os.path.join(dp, f))
    if not pdfs:
        years_with_no_pdf.append((year, "no_pdf"))
        continue

    yearly_data[year] = {"E":0.0, "S":0.0, "G":0.0, "UI":0.0, "total_tokens":0, "count":0}

    for fpath in pdfs:
        try:
            raw = read_pdf_with_fitz(fpath)
        except Exception as e:
            unreadable_items.append((year, fpath, str(e)))
            continue

        clean, tokens = preprocess(raw)
        total_tokens = len(tokens)
        if total_tokens == 0:
            continue

        E_cnt  = count_words(tokens, E_keywords) + count_phrases(clean, E_phrases)
        S_cnt  = count_words(tokens, S_keywords) + count_phrases(clean, S_phrases)
        G_cnt  = count_words(tokens, G_keywords) + count_phrases(clean, G_phrases)
        UI_cnt = count_words(tokens, UI_keywords)

        yd = yearly_data[year]
        yd["E"] += E_cnt
        yd["S"] += S_cnt
        yd["G"] += G_cnt
        yd["UI"] += UI_cnt
        yd["total_tokens"] += total_tokens
        yd["count"] += 1

# ====== 7) TÍNH AVERAGE ======
stats_rows = []
for year in YEARS:
    if year not in yearly_data:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue
    d = yearly_data[year]
    n, tok = d["count"], d["total_tokens"]
    if n == 0 or tok == 0:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue

    E_avg  = d["E"]/tok
    S_avg  = d["S"]/tok
    G_avg  = d["G"]/tok
    UI_avg = d["UI"]/tok

    stats_rows.append({"Year": year, "N_Files": n,
                       "E_avg": E_avg, "S_avg": S_avg, "G_avg": G_avg, "UI_avg": UI_avg})

df_stats = pd.DataFrame(stats_rows)

mask = df_stats["N_Files"] > 0
E_mu,  E_sd  = df_stats.loc[mask, "E_avg"].mean(), df_stats.loc[mask, "E_avg"].std(ddof=0)
S_mu,  S_sd  = df_stats.loc[mask, "S_avg"].mean(), df_stats.loc[mask, "S_avg"].std(ddof=0)
G_mu,  G_sd  = df_stats.loc[mask, "G_avg"].mean(), df_stats.loc[mask, "G_avg"].std(ddof=0)
UI_mu, UI_sd = df_stats.loc[mask, "UI_avg"].mean(), df_stats.loc[mask, "UI_avg"].std(ddof=0)

# ====== 😎 CHUẨN HÓA VỀ 0–100 ======
rows = []
for _, r in df_stats.iterrows():
    year = r["Year"]
    n    = r["N_Files"]
    if n == 0 or pd.isna(r["E_avg"]):
        rows.append({
            "Year": year, "N_Files": 0,
            "E_score_0_100": None, "S_score_0_100": None, "G_score_0_100": None, "UI_score_0_100": None,
            "ESG_subindex_0_100": None, "ESGUI_0_100": None
        })
        continue

    E_z, S_z, G_z, UI_z = (
        safe_z(r["E_avg"], E_mu, E_sd),
        safe_z(r["S_avg"], S_mu, S_sd),
        safe_z(r["G_avg"], G_mu, G_sd),
        safe_z(r["UI_avg"], UI_mu, UI_sd),
    )

    # convert Z to 0–100
    E_sc, S_sc, G_sc, UI_sc = (
        z_to_score(E_z),
        z_to_score(S_z),
        z_to_score(G_z),
        z_to_score(UI_z)
    )

    ESG_sub = round((E_sc + S_sc + G_sc)/3, 4)
    ESGUI   = round(0.5*ESG_sub + 0.5*UI_sc, 4)

    rows.append({
        "Year": year, "N_Files": int(n),
        "E_score_0_100": E_sc,
        "S_score_0_100": S_sc,
        "G_score_0_100": G_sc,
        "UI_score_0_100": UI_sc,
        "ESG_subindex_0_100": ESG_sub,
        "ESGUI_0_100": ESGUI
    })

df_yearly = pd.DataFrame(rows).sort_values("Year")
print("=== YEARLY SUMMARY (Z-score → 0–100) ===")
print(df_yearly.to_string(index=False))

# ====== 9) LOG ======
if years_with_no_pdf:
    print("\n-- Years missing data --")
    for y, typ in years_with_no_pdf:
        msg = "folder not found" if typ == "no_folder" else "no PDF found"
        print(f"{y}: {msg}")

if unreadable_items:
    print("\n-- Unreadable files --")
    for y, p, e in unreadable_items[:50]:
        print(f"[{y}] {p} -> {e}")
    if len(unreadable_items) > 50:
        print(f"... and {len(unreadable_items)-50} more")

print("\n--- Reference statistics ---")
print(f"E_mu={E_mu:.8f}, E_sd={E_sd:.8f}")
print(f"S_mu={S_mu:.8f}, S_sd={S_sd:.8f}")
print(f"G_mu={G_mu:.8f}, G_sd={G_sd:.8f}")
print(f"UI_mu={UI_mu:.8f}, UI_sd={UI_sd:.8f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MuPDF error: syntax error: unknown keyword: 'l1.759.25'

MuPDF error: syntax error: unknown keyword: '50F1.25'

MuPDF error: syntax error: unknown keyword: '50F1.2'

MuPDF error: syntax error: unknown keyword: '189.71.25'

MuPDF error: syntax error: unknown keyword: '50F1.2'

MuPDF error: syntax error: unknown keyword: '8l'

MuPDF error: syntax error: unknown keyword: '8l'

MuPDF error: syntax error: unknown keyword: '178..5'

MuPDF error: syntax error: unknown keyword: 'l189.716'

MuPDF error: syntax error: unknown keyword: 'l189.71'

MuPDF error: syntax error: unknown keyword: 'l318'

MuPDF error: syntax error: unknown keyword: 'l318'

MuPDF error: syntax error: unknown keyword: '0.8l4.25'

MuPDF error: syntax error: unknown keyword: '0.5020.7529'

MuPDF error: syntax error: unknown keyword: '0.82.255'

MuPDF error: syntax error: unknown keyword: 'll'

MuPD

In [ ]:
import numpy as np
import pandas as pd

# ---- 1) Chuẩn bị dữ liệu ----
japan = df = df_yearly.copy()

# ---- 2) Lấy 3 cột ESG Z→CDF→0–100 ----
ESG_cols = ["E_score_0_100", "S_score_0_100", "G_score_0_100"]

# ---- 3) DÙNG TRỰC TIẾP dữ liệu Z-score-0–100 cho entropy ----
X = df[ESG_cols].astype(float)

# Thêm 0.001 để tránh log(0) – GIỮ nguyên đúng công thức entropy
X_safe = X + 0.001

# ---- 4) p_ij = x_ij / sum(x_ij) (theo cột) ----
P = X_safe.div(X_safe.sum(axis=0), axis=1)

# ---- 5) Entropy ----
n = len(df)   # số năm
e = -(P * np.log(P)).sum(axis=0) / np.log(n)

# ---- 6) Divergence + Weights ----
d = 1 - e
w = d / d.sum()

print("Entropy weights (E, S, G):")
print(w.to_dict())

# ---- 7) Weighted ESG ----
df["E_weighted"] = df["E_score_0_100"] * w["E_score_0_100"]
df["S_weighted"] = df["S_score_0_100"] * w["S_score_0_100"]
df["G_weighted"] = df["G_score_0_100"] * w["G_score_0_100"]

df["ESG_sub_entropy"] = df["E_weighted"] + df["S_weighted"] + df["G_weighted"]

# ---- 😎 ESGUI mới ----
df["ESGUI_new"] = 0.5 * df["ESG_sub_entropy"] + 0.5 * df["UI_score_0_100"]

# ---- 9) Xuất bảng ----
pd.set_option('display.max_columns', None)

print(df[[
    "Year",
    "E_score_0_100", "S_score_0_100", "G_score_0_100",
    "E_weighted", "S_weighted", "G_weighted",
    "ESG_sub_entropy", "UI_score_0_100", "ESGUI_new"
]])

Entropy weights (E, S, G):
{'E_score_0_100': 0.27990627776874033, 'S_score_0_100': 0.32534743663531995, 'G_score_0_100': 0.3947462855959397}
    Year  E_score_0_100  S_score_0_100  G_score_0_100  E_weighted  S_weighted  \
0   2002         0.6208         0.5103         0.0409    0.173766    0.166025   
1   2003         0.3940         0.9894         0.1184    0.110283    0.321899   
2   2004         0.3308         0.9497         0.1254    0.092593    0.308982   
3   2005         0.4421         0.9862         0.3576    0.123747    0.320858   
4   2006         0.3250         0.7745         0.1553    0.090970    0.251982   
5   2007         0.9042         0.4111         0.1964    0.253091    0.133750   
6   2008         0.9864         0.5937         0.1406    0.276100    0.193159   
7   2009         0.5578         0.8351         0.3617    0.156132    0.271698   
8   2010         0.8428         0.0946         0.2547    0.235905    0.030778   
9   2011         0.9977         0.2447         0.

# VIETNAM

In [ ]:
##TRUNG QUỐC
##kw cũ
#zcore
import os, re
import fitz  # PyMuPDF
import nltk
import pandas as pd
import numpy as np
from scipy.stats import norm
from nltk.tokenize import word_tokenize
from collections import Counter

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# NLTK setup
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# ====== 1) ĐỌC PDF & TIỀN XỬ LÝ ======
def read_pdf_with_fitz(path):
    doc = fitz.open(path)
    text = []
    for page in doc:
        text.append(page.get_text("text"))
    return "\n".join(text)

def preprocess(raw):
    clean = raw.lower()
    clean = clean.replace("co₂", "co2").replace("so₂", "so2")
    clean = re.sub(r'[^a-z\s]', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    tokens = word_tokenize(clean)
    return clean, tokens

# ====== 2) TỪ KHÓA/CỤM TỪ ======
E_keywords = [
    "acid","agriculture","biodiversity","carbon","clean","co2","conservation","deforestation",
    "depletion","drought","ecofriendly","ecology","emission","energy","erosion","flood","footprint",
    "fossil","geothermal","green","greenhouse","hydro","landfill","methane","microplastics","monoxide",
    "natural","noise","oil","organic","overconsumption","overfishing","overpopulation","ozone",
    "pesticide","plastic","poison","pollution","radiation","recycling","renewable","resources",
    "so2","smog","soil","solar","storm","sustainability","toxicity","trash","waste","wastewater","wind"
]
E_phrases = ["carbon footprint","carbon emission","renewable energy","wind energy","greenhouse gas"]

S_keywords = [
    "agility","career","charitableness","collaboration","communication","conscience","corruption","crime",
    "culture","dialogue","discrimination","donation","donations","elderly","empathy","employment",
    "entrepreneurship","equal","female","flexibility","freedom","gender","harassment","harmony","honesty",
    "humanity","innovation","integration","mentorship","morality","participation","profession","respect",
    "responsibility","health","community","inclusion","wellbeing","safety","privacy","labor"
]
S_phrases = ["gender equality","labor rights","human rights","data privacy","workplace safety","community development"]

G_keywords = [
    "accountability","adaptability","agility","audit","autonomy","board","coherence","collaboration",
    "communication","compensation","compliance","consistency","corporate","culture","corruption","data",
    "protection","decisions","diligence","diversity","efficiency","ethics","evaluation","fairness",
    "leadership","legitimacy","meritocracy","objectivity","oversight","performance","principle",
    "prudence","quality","reciprocity","relations","reliability","resilience","responsibility","risk",
    "stakeholders","stewardship","strategy","sustainability","transparency"
]
G_phrases = ["board independence","executive compensation","shareholder rights","stakeholder engagement",
             "anti corruption","anti-corruption","data protection"]
UI_keywords = ["uncertain","uncertainty","uncertainties","risk","risks","volatile","volatility","instability","ambiguity","fluctuation","doubt"]

# ====== 3) HÀM ĐẾM ======
def count_words(tokens, vocab):
    vocab = set(w.lower() for w in vocab)
    c = Counter(tokens)
    return sum(c[w] for w in vocab)

def count_phrases(clean_text, phrases):
    total = 0
    for p in phrases:
        pat = r'\b' + re.sub(r'\s+', r'\\s+', re.escape(p.lower())) + r'\b'
        total += len(re.findall(pat, clean_text))
    return total

# ====== 4) SAFE Z + CDF chuyển về 0–100 ======
def safe_z(x, mu, sigma):
    return 0 if sigma == 0 or np.isnan(sigma) else (x - mu) / sigma

def z_to_score(z):
    """Convert Z to 0–100 scale using Normal CDF"""
    return round(norm.cdf(z), 4)

# ====== 5) CẤU HÌNH ======
BASE_DIR = '/content/drive/MyDrive/PETTIES/DATA BÁO CÁO - nguồn để tính EW_ESGUI/VIETNAM 2'
YEARS = [str(y) for y in range(2002, 2025)]

yearly_data = {}
years_with_no_pdf = []
unreadable_items = []

# ====== 6) ĐỌC PDF THEO NĂM ======
for year in YEARS:
    year_dir = os.path.join(BASE_DIR, year)
    if not os.path.isdir(year_dir):
        years_with_no_pdf.append((year, "no_folder"))
        continue

    pdfs = []
    for dp, dn, fn in os.walk(year_dir):
        for f in fn:
            if f.lower().endswith(".pdf"):
                pdfs.append(os.path.join(dp, f))
    if not pdfs:
        years_with_no_pdf.append((year, "no_pdf"))
        continue

    yearly_data[year] = {"E":0.0, "S":0.0, "G":0.0, "UI":0.0, "total_tokens":0, "count":0}

    for fpath in pdfs:
        try:
            raw = read_pdf_with_fitz(fpath)
        except Exception as e:
            unreadable_items.append((year, fpath, str(e)))
            continue

        clean, tokens = preprocess(raw)
        total_tokens = len(tokens)
        if total_tokens == 0:
            continue

        E_cnt  = count_words(tokens, E_keywords) + count_phrases(clean, E_phrases)
        S_cnt  = count_words(tokens, S_keywords) + count_phrases(clean, S_phrases)
        G_cnt  = count_words(tokens, G_keywords) + count_phrases(clean, G_phrases)
        UI_cnt = count_words(tokens, UI_keywords)

        yd = yearly_data[year]
        yd["E"] += E_cnt
        yd["S"] += S_cnt
        yd["G"] += G_cnt
        yd["UI"] += UI_cnt
        yd["total_tokens"] += total_tokens
        yd["count"] += 1

# ====== 7) TÍNH AVERAGE ======
stats_rows = []
for year in YEARS:
    if year not in yearly_data:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue
    d = yearly_data[year]
    n, tok = d["count"], d["total_tokens"]
    if n == 0 or tok == 0:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue

    E_avg  = d["E"]/tok
    S_avg  = d["S"]/tok
    G_avg  = d["G"]/tok
    UI_avg = d["UI"]/tok

    stats_rows.append({"Year": year, "N_Files": n,
                       "E_avg": E_avg, "S_avg": S_avg, "G_avg": G_avg, "UI_avg": UI_avg})

df_stats = pd.DataFrame(stats_rows)

mask = df_stats["N_Files"] > 0
E_mu,  E_sd  = df_stats.loc[mask, "E_avg"].mean(), df_stats.loc[mask, "E_avg"].std(ddof=0)
S_mu,  S_sd  = df_stats.loc[mask, "S_avg"].mean(), df_stats.loc[mask, "S_avg"].std(ddof=0)
G_mu,  G_sd  = df_stats.loc[mask, "G_avg"].mean(), df_stats.loc[mask, "G_avg"].std(ddof=0)
UI_mu, UI_sd = df_stats.loc[mask, "UI_avg"].mean(), df_stats.loc[mask, "UI_avg"].std(ddof=0)

# ====== 😎 CHUẨN HÓA VỀ 0–100 ======
rows = []
for _, r in df_stats.iterrows():
    year = r["Year"]
    n    = r["N_Files"]
    if n == 0 or pd.isna(r["E_avg"]):
        rows.append({
            "Year": year, "N_Files": 0,
            "E_score_0_100": None, "S_score_0_100": None, "G_score_0_100": None, "UI_score_0_100": None,
            "ESG_subindex_0_100": None, "ESGUI_0_100": None
        })
        continue

    E_z, S_z, G_z, UI_z = (
        safe_z(r["E_avg"], E_mu, E_sd),
        safe_z(r["S_avg"], S_mu, S_sd),
        safe_z(r["G_avg"], G_mu, G_sd),
        safe_z(r["UI_avg"], UI_mu, UI_sd),
    )

    # convert Z to 0–100
    E_sc, S_sc, G_sc, UI_sc = (
        z_to_score(E_z),
        z_to_score(S_z),
        z_to_score(G_z),
        z_to_score(UI_z)
    )

    ESG_sub = round((E_sc + S_sc + G_sc)/3, 4)
    ESGUI   = round(0.5*ESG_sub + 0.5*UI_sc, 4)

    rows.append({
        "Year": year, "N_Files": int(n),
        "E_score_0_100": E_sc,
        "S_score_0_100": S_sc,
        "G_score_0_100": G_sc,
        "UI_score_0_100": UI_sc,
        "ESG_subindex_0_100": ESG_sub,
        "ESGUI_0_100": ESGUI
    })

df_yearly = pd.DataFrame(rows).sort_values("Year")
print("=== YEARLY SUMMARY (Z-score → 0–100) ===")
print(df_yearly.to_string(index=False))

# ====== 9) LOG ======
if years_with_no_pdf:
    print("\n-- Years missing data --")
    for y, typ in years_with_no_pdf:
        msg = "folder not found" if typ == "no_folder" else "no PDF found"
        print(f"{y}: {msg}")

if unreadable_items:
    print("\n-- Unreadable files --")
    for y, p, e in unreadable_items[:50]:
        print(f"[{y}] {p} -> {e}")
    if len(unreadable_items) > 50:
        print(f"... and {len(unreadable_items)-50} more")

print("\n--- Reference statistics ---")
print(f"E_mu={E_mu:.8f}, E_sd={E_sd:.8f}")
print(f"S_mu={S_mu:.8f}, S_sd={S_sd:.8f}")
print(f"G_mu={G_mu:.8f}, G_sd={G_sd:.8f}")
print(f"UI_mu={UI_mu:.8f}, UI_sd={UI_sd:.8f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

=== YEARLY SUMMARY (Z-score → 0–100) ===
Year  N_Files  E_score_0_100  S_score_0_100  G_score_0_100  UI_score_0_100  ESG_subindex_0_100  ESGUI_0_100
2002       14         0.2281         0.8086         0.0963          0.1480              0.3777       0.2628
2003       14         0.8929         0.0386         0.3192          0.1305              0.4169       0.2737
2004       12         0.9979         0.0697         0.0713          0.1792              0.3796       0.2794
2005       12         0.7226         0.4237         0.1249          0.0917              0.4237       0.2577
2006       12         0.8067         0.1438         0.7105          0.0955              0.5537       0.3246
2007       13         0.9288         0.0197         0.0853          

In [ ]:
import numpy as np
import pandas as pd

# ---- 1) Chuẩn bị dữ liệu ----
vietnam = df = df_yearly.copy()

# ---- 2) Lấy 3 cột ESG Z→CDF→0–100 ----
ESG_cols = ["E_score_0_100", "S_score_0_100", "G_score_0_100"]

# ---- 3) DÙNG TRỰC TIẾP dữ liệu Z-score-0–100 cho entropy ----
X = df[ESG_cols].astype(float)

# Thêm 0.001 để tránh log(0) – GIỮ nguyên đúng công thức entropy
X_safe = X + 0.001

# ---- 4) p_ij = x_ij / sum(x_ij) (theo cột) ----
P = X_safe.div(X_safe.sum(axis=0), axis=1)

# ---- 5) Entropy ----
n = len(df)   # số năm
e = -(P * np.log(P)).sum(axis=0) / np.log(n)

# ---- 6) Divergence + Weights ----
d = 1 - e
w = d / d.sum()

print("Entropy weights (E, S, G):")
print(w.to_dict())

# ---- 7) Weighted ESG ----
df["E_weighted"] = df["E_score_0_100"] * w["E_score_0_100"]
df["S_weighted"] = df["S_score_0_100"] * w["S_score_0_100"]
df["G_weighted"] = df["G_score_0_100"] * w["G_score_0_100"]

df["ESG_sub_entropy"] = df["E_weighted"] + df["S_weighted"] + df["G_weighted"]

# ---- 😎 ESGUI mới ----
df["ESGUI_new"] = 0.5 * df["ESG_sub_entropy"] + 0.5 * df["UI_score_0_100"]

# ---- 9) Xuất bảng ----
pd.set_option('display.max_columns', None)

print(df[[
    "Year",
    "E_score_0_100", "S_score_0_100", "G_score_0_100",
    "E_weighted", "S_weighted", "G_weighted",
    "ESG_sub_entropy", "UI_score_0_100", "ESGUI_new"
]])

Entropy weights (E, S, G):
{'E_score_0_100': 0.31001185064847103, 'S_score_0_100': 0.34003922042847995, 'G_score_0_100': 0.3499489289230491}
    Year  E_score_0_100  S_score_0_100  G_score_0_100  E_weighted  S_weighted  \
0   2002         0.2281         0.8086         0.0963    0.070714    0.274956   
1   2003         0.8929         0.0386         0.3192    0.276810    0.013126   
2   2004         0.9979         0.0697         0.0713    0.309361    0.023701   
3   2005         0.7226         0.4237         0.1249    0.224015    0.144075   
4   2006         0.8067         0.1438         0.7105    0.250087    0.048898   
5   2007         0.9288         0.0197         0.0853    0.287939    0.006699   
6   2008         0.8138         0.6618         0.3305    0.252288    0.225038   
7   2009         0.8701         0.0733         0.9103    0.269741    0.024925   
8   2010         0.2603         0.8301         0.9671    0.080696    0.282267   
9   2011         0.7286         0.3856         0.

# SINGAPORE

In [ ]:
##TRUNG QUỐC
##kw cũ
#zcore
import os, re
import fitz  # PyMuPDF
import nltk
import pandas as pd
import numpy as np
from scipy.stats import norm
from nltk.tokenize import word_tokenize
from collections import Counter

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# NLTK setup
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# ====== 1) ĐỌC PDF & TIỀN XỬ LÝ ======
def read_pdf_with_fitz(path):
    doc = fitz.open(path)
    text = []
    for page in doc:
        text.append(page.get_text("text"))
    return "\n".join(text)

def preprocess(raw):
    clean = raw.lower()
    clean = clean.replace("co₂", "co2").replace("so₂", "so2")
    clean = re.sub(r'[^a-z\s]', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    tokens = word_tokenize(clean)
    return clean, tokens

# ====== 2) TỪ KHÓA/CỤM TỪ ======
E_keywords = [
    "acid","agriculture","biodiversity","carbon","clean","co2","conservation","deforestation",
    "depletion","drought","ecofriendly","ecology","emission","energy","erosion","flood","footprint",
    "fossil","geothermal","green","greenhouse","hydro","landfill","methane","microplastics","monoxide",
    "natural","noise","oil","organic","overconsumption","overfishing","overpopulation","ozone",
    "pesticide","plastic","poison","pollution","radiation","recycling","renewable","resources",
    "so2","smog","soil","solar","storm","sustainability","toxicity","trash","waste","wastewater","wind"
]
E_phrases = ["carbon footprint","carbon emission","renewable energy","wind energy","greenhouse gas"]

S_keywords = [
    "agility","career","charitableness","collaboration","communication","conscience","corruption","crime",
    "culture","dialogue","discrimination","donation","donations","elderly","empathy","employment",
    "entrepreneurship","equal","female","flexibility","freedom","gender","harassment","harmony","honesty",
    "humanity","innovation","integration","mentorship","morality","participation","profession","respect",
    "responsibility","health","community","inclusion","wellbeing","safety","privacy","labor"
]
S_phrases = ["gender equality","labor rights","human rights","data privacy","workplace safety","community development"]

G_keywords = [
    "accountability","adaptability","agility","audit","autonomy","board","coherence","collaboration",
    "communication","compensation","compliance","consistency","corporate","culture","corruption","data",
    "protection","decisions","diligence","diversity","efficiency","ethics","evaluation","fairness",
    "leadership","legitimacy","meritocracy","objectivity","oversight","performance","principle",
    "prudence","quality","reciprocity","relations","reliability","resilience","responsibility","risk",
    "stakeholders","stewardship","strategy","sustainability","transparency"
]
G_phrases = ["board independence","executive compensation","shareholder rights","stakeholder engagement",
             "anti corruption","anti-corruption","data protection"]
UI_keywords = ["uncertain","uncertainty","uncertainties","risk","risks","volatile","volatility","instability","ambiguity","fluctuation","doubt"]

# ====== 3) HÀM ĐẾM ======
def count_words(tokens, vocab):
    vocab = set(w.lower() for w in vocab)
    c = Counter(tokens)
    return sum(c[w] for w in vocab)

def count_phrases(clean_text, phrases):
    total = 0
    for p in phrases:
        pat = r'\b' + re.sub(r'\s+', r'\\s+', re.escape(p.lower())) + r'\b'
        total += len(re.findall(pat, clean_text))
    return total

# ====== 4) SAFE Z + CDF chuyển về 0–100 ======
def safe_z(x, mu, sigma):
    return 0 if sigma == 0 or np.isnan(sigma) else (x - mu) / sigma

def z_to_score(z):
    """Convert Z to 0–100 scale using Normal CDF"""
    return round(norm.cdf(z), 4)

# ====== 5) CẤU HÌNH ======
BASE_DIR = '/content/drive/MyDrive/PETTIES/DATA BÁO CÁO - nguồn để tính EW_ESGUI/Sing two'
YEARS = [str(y) for y in range(2002, 2025)]

yearly_data = {}
years_with_no_pdf = []
unreadable_items = []

# ====== 6) ĐỌC PDF THEO NĂM ======
for year in YEARS:
    year_dir = os.path.join(BASE_DIR, year)
    if not os.path.isdir(year_dir):
        years_with_no_pdf.append((year, "no_folder"))
        continue

    pdfs = []
    for dp, dn, fn in os.walk(year_dir):
        for f in fn:
            if f.lower().endswith(".pdf"):
                pdfs.append(os.path.join(dp, f))
    if not pdfs:
        years_with_no_pdf.append((year, "no_pdf"))
        continue

    yearly_data[year] = {"E":0.0, "S":0.0, "G":0.0, "UI":0.0, "total_tokens":0, "count":0}

    for fpath in pdfs:
        try:
            raw = read_pdf_with_fitz(fpath)
        except Exception as e:
            unreadable_items.append((year, fpath, str(e)))
            continue

        clean, tokens = preprocess(raw)
        total_tokens = len(tokens)
        if total_tokens == 0:
            continue

        E_cnt  = count_words(tokens, E_keywords) + count_phrases(clean, E_phrases)
        S_cnt  = count_words(tokens, S_keywords) + count_phrases(clean, S_phrases)
        G_cnt  = count_words(tokens, G_keywords) + count_phrases(clean, G_phrases)
        UI_cnt = count_words(tokens, UI_keywords)

        yd = yearly_data[year]
        yd["E"] += E_cnt
        yd["S"] += S_cnt
        yd["G"] += G_cnt
        yd["UI"] += UI_cnt
        yd["total_tokens"] += total_tokens
        yd["count"] += 1

# ====== 7) TÍNH AVERAGE ======
stats_rows = []
for year in YEARS:
    if year not in yearly_data:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue
    d = yearly_data[year]
    n, tok = d["count"], d["total_tokens"]
    if n == 0 or tok == 0:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue

    E_avg  = d["E"]/tok
    S_avg  = d["S"]/tok
    G_avg  = d["G"]/tok
    UI_avg = d["UI"]/tok

    stats_rows.append({"Year": year, "N_Files": n,
                       "E_avg": E_avg, "S_avg": S_avg, "G_avg": G_avg, "UI_avg": UI_avg})

df_stats = pd.DataFrame(stats_rows)

mask = df_stats["N_Files"] > 0
E_mu,  E_sd  = df_stats.loc[mask, "E_avg"].mean(), df_stats.loc[mask, "E_avg"].std(ddof=0)
S_mu,  S_sd  = df_stats.loc[mask, "S_avg"].mean(), df_stats.loc[mask, "S_avg"].std(ddof=0)
G_mu,  G_sd  = df_stats.loc[mask, "G_avg"].mean(), df_stats.loc[mask, "G_avg"].std(ddof=0)
UI_mu, UI_sd = df_stats.loc[mask, "UI_avg"].mean(), df_stats.loc[mask, "UI_avg"].std(ddof=0)

# ====== 😎 CHUẨN HÓA VỀ 0–100 ======
rows = []
for _, r in df_stats.iterrows():
    year = r["Year"]
    n    = r["N_Files"]
    if n == 0 or pd.isna(r["E_avg"]):
        rows.append({
            "Year": year, "N_Files": 0,
            "E_score_0_100": None, "S_score_0_100": None, "G_score_0_100": None, "UI_score_0_100": None,
            "ESG_subindex_0_100": None, "ESGUI_0_100": None
        })
        continue

    E_z, S_z, G_z, UI_z = (
        safe_z(r["E_avg"], E_mu, E_sd),
        safe_z(r["S_avg"], S_mu, S_sd),
        safe_z(r["G_avg"], G_mu, G_sd),
        safe_z(r["UI_avg"], UI_mu, UI_sd),
    )

    # convert Z to 0–100
    E_sc, S_sc, G_sc, UI_sc = (
        z_to_score(E_z),
        z_to_score(S_z),
        z_to_score(G_z),
        z_to_score(UI_z)
    )

    ESG_sub = round((E_sc + S_sc + G_sc)/3, 4)
    ESGUI   = round(0.5*ESG_sub + 0.5*UI_sc, 4)

    rows.append({
        "Year": year, "N_Files": int(n),
        "E_score_0_100": E_sc,
        "S_score_0_100": S_sc,
        "G_score_0_100": G_sc,
        "UI_score_0_100": UI_sc,
        "ESG_subindex_0_100": ESG_sub,
        "ESGUI_0_100": ESGUI
    })

df_yearly = pd.DataFrame(rows).sort_values("Year")
print("=== YEARLY SUMMARY (Z-score → 0–100) ===")
print(df_yearly.to_string(index=False))

# ====== 9) LOG ======
if years_with_no_pdf:
    print("\n-- Years missing data --")
    for y, typ in years_with_no_pdf:
        msg = "folder not found" if typ == "no_folder" else "no PDF found"
        print(f"{y}: {msg}")

if unreadable_items:
    print("\n-- Unreadable files --")
    for y, p, e in unreadable_items[:50]:
        print(f"[{y}] {p} -> {e}")
    if len(unreadable_items) > 50:
        print(f"... and {len(unreadable_items)-50} more")

print("\n--- Reference statistics ---")
print(f"E_mu={E_mu:.8f}, E_sd={E_sd:.8f}")
print(f"S_mu={S_mu:.8f}, S_sd={S_sd:.8f}")
print(f"G_mu={G_mu:.8f}, G_sd={G_sd:.8f}")
print(f"UI_mu={UI_mu:.8f}, UI_sd={UI_sd:.8f}")

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object in xref (94 0 R)

MuPDF error: format error: cannot find object 

In [ ]:
import numpy as np
import pandas as pd

# ---- 1) Chuẩn bị dữ liệu ----
singapore = df = df_yearly.copy()

# ---- 2) Lấy 3 cột ESG Z→CDF→0–100 ----
ESG_cols = ["E_score_0_100", "S_score_0_100", "G_score_0_100"]

# ---- 3) DÙNG TRỰC TIẾP dữ liệu Z-score-0–100 cho entropy ----
X = df[ESG_cols].astype(float)

# Thêm 0.001 để tránh log(0) – GIỮ nguyên đúng công thức entropy
X_safe = X + 0.001

# ---- 4) p_ij = x_ij / sum(x_ij) (theo cột) ----
P = X_safe.div(X_safe.sum(axis=0), axis=1)

# ---- 5) Entropy ----
n = len(df)   # số năm
e = -(P * np.log(P)).sum(axis=0) / np.log(n)

# ---- 6) Divergence + Weights ----
d = 1 - e
w = d / d.sum()

print("Entropy weights (E, S, G):")
print(w.to_dict())

# ---- 7) Weighted ESG ----
df["E_weighted"] = df["E_score_0_100"] * w["E_score_0_100"]
df["S_weighted"] = df["S_score_0_100"] * w["S_score_0_100"]
df["G_weighted"] = df["G_score_0_100"] * w["G_score_0_100"]

df["ESG_sub_entropy"] = df["E_weighted"] + df["S_weighted"] + df["G_weighted"]

# ---- 😎 ESGUI mới ----
df["ESGUI_new"] = 0.5 * df["ESG_sub_entropy"] + 0.5 * df["UI_score_0_100"]

# ---- 9) Xuất bảng ----
pd.set_option('display.max_columns', None)

print(df[[
    "Year",
    "E_score_0_100", "S_score_0_100", "G_score_0_100",
    "E_weighted", "S_weighted", "G_weighted",
    "ESG_sub_entropy", "UI_score_0_100", "ESGUI_new"
]])

Entropy weights (E, S, G):
{'E_score_0_100': 0.3408055925845536, 'S_score_0_100': 0.3617438319758608, 'G_score_0_100': 0.2974505754395856}
    Year  E_score_0_100  S_score_0_100  G_score_0_100  E_weighted  S_weighted  \
0   2002         0.1177         0.0705         0.3237    0.040113    0.025503   
1   2003         0.1565         0.3600         0.1020    0.053336    0.130228   
2   2004         0.3108         0.0911         0.0098    0.105922    0.032955   
3   2005         0.9623         0.0890         0.3175    0.327957    0.032195   
4   2006         0.2556         0.2279         0.4739    0.087110    0.082441   
5   2007         0.2664         0.3471         0.3910    0.090791    0.125561   
6   2008         0.9620         0.1137         0.2514    0.327855    0.041130   
7   2009         0.9315         0.1044         0.2601    0.317460    0.037766   
8   2010         0.9153         0.0496         0.2899    0.311939    0.017942   
9   2011         0.9556         0.5733         0.35

# KOREA

In [ ]:
##TRUNG QUỐC
##kw cũ
#zcore
import os, re
import fitz  # PyMuPDF
import nltk
import pandas as pd
import numpy as np
from scipy.stats import norm
from nltk.tokenize import word_tokenize
from collections import Counter

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# NLTK setup
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# ====== 1) ĐỌC PDF & TIỀN XỬ LÝ ======
def read_pdf_with_fitz(path):
    doc = fitz.open(path)
    text = []
    for page in doc:
        text.append(page.get_text("text"))
    return "\n".join(text)

def preprocess(raw):
    clean = raw.lower()
    clean = clean.replace("co₂", "co2").replace("so₂", "so2")
    clean = re.sub(r'[^a-z\s]', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    tokens = word_tokenize(clean)
    return clean, tokens

# ====== 2) TỪ KHÓA/CỤM TỪ ======
E_keywords = [
    "acid","agriculture","biodiversity","carbon","clean","co2","conservation","deforestation",
    "depletion","drought","ecofriendly","ecology","emission","energy","erosion","flood","footprint",
    "fossil","geothermal","green","greenhouse","hydro","landfill","methane","microplastics","monoxide",
    "natural","noise","oil","organic","overconsumption","overfishing","overpopulation","ozone",
    "pesticide","plastic","poison","pollution","radiation","recycling","renewable","resources",
    "so2","smog","soil","solar","storm","sustainability","toxicity","trash","waste","wastewater","wind"
]
E_phrases = ["carbon footprint","carbon emission","renewable energy","wind energy","greenhouse gas"]

S_keywords = [
    "agility","career","charitableness","collaboration","communication","conscience","corruption","crime",
    "culture","dialogue","discrimination","donation","donations","elderly","empathy","employment",
    "entrepreneurship","equal","female","flexibility","freedom","gender","harassment","harmony","honesty",
    "humanity","innovation","integration","mentorship","morality","participation","profession","respect",
    "responsibility","health","community","inclusion","wellbeing","safety","privacy","labor"
]
S_phrases = ["gender equality","labor rights","human rights","data privacy","workplace safety","community development"]

G_keywords = [
    "accountability","adaptability","agility","audit","autonomy","board","coherence","collaboration",
    "communication","compensation","compliance","consistency","corporate","culture","corruption","data",
    "protection","decisions","diligence","diversity","efficiency","ethics","evaluation","fairness",
    "leadership","legitimacy","meritocracy","objectivity","oversight","performance","principle",
    "prudence","quality","reciprocity","relations","reliability","resilience","responsibility","risk",
    "stakeholders","stewardship","strategy","sustainability","transparency"
]
G_phrases = ["board independence","executive compensation","shareholder rights","stakeholder engagement",
             "anti corruption","anti-corruption","data protection"]
UI_keywords = ["uncertain","uncertainty","uncertainties","risk","risks","volatile","volatility","instability","ambiguity","fluctuation","doubt"]

# ====== 3) HÀM ĐẾM ======
def count_words(tokens, vocab):
    vocab = set(w.lower() for w in vocab)
    c = Counter(tokens)
    return sum(c[w] for w in vocab)

def count_phrases(clean_text, phrases):
    total = 0
    for p in phrases:
        pat = r'\b' + re.sub(r'\s+', r'\\s+', re.escape(p.lower())) + r'\b'
        total += len(re.findall(pat, clean_text))
    return total

# ====== 4) SAFE Z + CDF chuyển về 0–100 ======
def safe_z(x, mu, sigma):
    return 0 if sigma == 0 or np.isnan(sigma) else (x - mu) / sigma

def z_to_score(z):
    """Convert Z to 0–100 scale using Normal CDF"""
    return round(norm.cdf(z), 4)

# ====== 5) CẤU HÌNH ======
BASE_DIR = '/content/drive/MyDrive/PETTIES/DATA BÁO CÁO - nguồn để tính EW_ESGUI/Hàn quắc'
YEARS = [str(y) for y in range(2002, 2025)]

yearly_data = {}
years_with_no_pdf = []
unreadable_items = []

# ====== 6) ĐỌC PDF THEO NĂM ======
for year in YEARS:
    year_dir = os.path.join(BASE_DIR, year)
    if not os.path.isdir(year_dir):
        years_with_no_pdf.append((year, "no_folder"))
        continue

    pdfs = []
    for dp, dn, fn in os.walk(year_dir):
        for f in fn:
            if f.lower().endswith(".pdf"):
                pdfs.append(os.path.join(dp, f))
    if not pdfs:
        years_with_no_pdf.append((year, "no_pdf"))
        continue

    yearly_data[year] = {"E":0.0, "S":0.0, "G":0.0, "UI":0.0, "total_tokens":0, "count":0}

    for fpath in pdfs:
        try:
            raw = read_pdf_with_fitz(fpath)
        except Exception as e:
            unreadable_items.append((year, fpath, str(e)))
            continue

        clean, tokens = preprocess(raw)
        total_tokens = len(tokens)
        if total_tokens == 0:
            continue

        E_cnt  = count_words(tokens, E_keywords) + count_phrases(clean, E_phrases)
        S_cnt  = count_words(tokens, S_keywords) + count_phrases(clean, S_phrases)
        G_cnt  = count_words(tokens, G_keywords) + count_phrases(clean, G_phrases)
        UI_cnt = count_words(tokens, UI_keywords)

        yd = yearly_data[year]
        yd["E"] += E_cnt
        yd["S"] += S_cnt
        yd["G"] += G_cnt
        yd["UI"] += UI_cnt
        yd["total_tokens"] += total_tokens
        yd["count"] += 1

# ====== 7) TÍNH AVERAGE ======
stats_rows = []
for year in YEARS:
    if year not in yearly_data:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue
    d = yearly_data[year]
    n, tok = d["count"], d["total_tokens"]
    if n == 0 or tok == 0:
        stats_rows.append({"Year": year, "N_Files": 0,
                           "E_avg": None, "S_avg": None, "G_avg": None, "UI_avg": None})
        continue

    E_avg  = d["E"]/tok
    S_avg  = d["S"]/tok
    G_avg  = d["G"]/tok
    UI_avg = d["UI"]/tok

    stats_rows.append({"Year": year, "N_Files": n,
                       "E_avg": E_avg, "S_avg": S_avg, "G_avg": G_avg, "UI_avg": UI_avg})

df_stats = pd.DataFrame(stats_rows)

mask = df_stats["N_Files"] > 0
E_mu,  E_sd  = df_stats.loc[mask, "E_avg"].mean(), df_stats.loc[mask, "E_avg"].std(ddof=0)
S_mu,  S_sd  = df_stats.loc[mask, "S_avg"].mean(), df_stats.loc[mask, "S_avg"].std(ddof=0)
G_mu,  G_sd  = df_stats.loc[mask, "G_avg"].mean(), df_stats.loc[mask, "G_avg"].std(ddof=0)
UI_mu, UI_sd = df_stats.loc[mask, "UI_avg"].mean(), df_stats.loc[mask, "UI_avg"].std(ddof=0)

# ====== 😎 CHUẨN HÓA VỀ 0–100 ======
rows = []
for _, r in df_stats.iterrows():
    year = r["Year"]
    n    = r["N_Files"]
    if n == 0 or pd.isna(r["E_avg"]):
        rows.append({
            "Year": year, "N_Files": 0,
            "E_score_0_100": None, "S_score_0_100": None, "G_score_0_100": None, "UI_score_0_100": None,
            "ESG_subindex_0_100": None, "ESGUI_0_100": None
        })
        continue

    E_z, S_z, G_z, UI_z = (
        safe_z(r["E_avg"], E_mu, E_sd),
        safe_z(r["S_avg"], S_mu, S_sd),
        safe_z(r["G_avg"], G_mu, G_sd),
        safe_z(r["UI_avg"], UI_mu, UI_sd),
    )

    # convert Z to 0–100
    E_sc, S_sc, G_sc, UI_sc = (
        z_to_score(E_z),
        z_to_score(S_z),
        z_to_score(G_z),
        z_to_score(UI_z)
    )

    ESG_sub = round((E_sc + S_sc + G_sc)/3, 4)
    ESGUI   = round(0.5*ESG_sub + 0.5*UI_sc, 4)

    rows.append({
        "Year": year, "N_Files": int(n),
        "E_score_0_100": E_sc,
        "S_score_0_100": S_sc,
        "G_score_0_100": G_sc,
        "UI_score_0_100": UI_sc,
        "ESG_subindex_0_100": ESG_sub,
        "ESGUI_0_100": ESGUI
    })

df_yearly = pd.DataFrame(rows).sort_values("Year")
print("=== YEARLY SUMMARY (Z-score → 0–100) ===")
print(df_yearly.to_string(index=False))

# ====== 9) LOG ======
if years_with_no_pdf:
    print("\n-- Years missing data --")
    for y, typ in years_with_no_pdf:
        msg = "folder not found" if typ == "no_folder" else "no PDF found"
        print(f"{y}: {msg}")

if unreadable_items:
    print("\n-- Unreadable files --")
    for y, p, e in unreadable_items[:50]:
        print(f"[{y}] {p} -> {e}")
    if len(unreadable_items) > 50:
        print(f"... and {len(unreadable_items)-50} more")

print("\n--- Reference statistics ---")
print(f"E_mu={E_mu:.8f}, E_sd={E_sd:.8f}")
print(f"S_mu={S_mu:.8f}, S_sd={S_sd:.8f}")
print(f"G_mu={G_mu:.8f}, G_sd={G_sd:.8f}")
print(f"UI_mu={UI_mu:.8f}, UI_sd={UI_sd:.8f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

=== YEARLY SUMMARY (Z-score → 0–100) ===
Year  N_Files  E_score_0_100  S_score_0_100  G_score_0_100  UI_score_0_100  ESG_subindex_0_100  ESGUI_0_100
2002        6         0.2131         0.0489         0.1394          0.2293              0.1338       0.1816
2003        6         0.0936         0.3200         0.3286          0.4007              0.2474       0.3240
2004        4         0.0414         0.2690         0.2343          0.3993              0.1816       0.2904
2005        5         0.1212         0.9200         0.3587          0.3595              0.4666       0.4130
2006        9         0.9229         0.1174         0.6568          0.1838              0.5657       0.3747
2007        6         0.0924         0.3754         0.0356          0.2834    

In [ ]:
import numpy as np
import pandas as pd

# ---- 1) Chuẩn bị dữ liệu ----
korea = df = df_yearly.copy()

# ---- 2) Lấy 3 cột ESG Z→CDF→0–100 ----
ESG_cols = ["E_score_0_100", "S_score_0_100", "G_score_0_100"]

# ---- 3) DÙNG TRỰC TIẾP dữ liệu Z-score-0–100 cho entropy ----
X = df[ESG_cols].astype(float)

# Thêm 0.001 để tránh log(0) – GIỮ nguyên đúng công thức entropy
X_safe = X + 0.001

# ---- 4) p_ij = x_ij / sum(x_ij) (theo cột) ----
P = X_safe.div(X_safe.sum(axis=0), axis=1)

# ---- 5) Entropy ----
n = len(df)   # số năm
e = -(P * np.log(P)).sum(axis=0) / np.log(n)

# ---- 6) Divergence + Weights ----
d = 1 - e
w = d / d.sum()

print("Entropy weights (E, S, G):")
print(w.to_dict())

# ---- 7) Weighted ESG ----
df["E_weighted"] = df["E_score_0_100"] * w["E_score_0_100"]
df["S_weighted"] = df["S_score_0_100"] * w["S_score_0_100"]
df["G_weighted"] = df["G_score_0_100"] * w["G_score_0_100"]

df["ESG_sub_entropy"] = df["E_weighted"] + df["S_weighted"] + df["G_weighted"]

# ---- 😎 ESGUI mới ----
df["ESGUI_new"] = 0.5 * df["ESG_sub_entropy"] + 0.5 * df["UI_score_0_100"]

# ---- 9) Xuất bảng ----
pd.set_option('display.max_columns', None)

print(df[[
    "Year",
    "E_score_0_100", "S_score_0_100", "G_score_0_100",
    "E_weighted", "S_weighted", "G_weighted",
    "ESG_sub_entropy", "UI_score_0_100", "ESGUI_new"
]])

Entropy weights (E, S, G):
{'E_score_0_100': 0.3772454898220189, 'S_score_0_100': 0.3558378098514648, 'G_score_0_100': 0.2669167003265163}
    Year  E_score_0_100  S_score_0_100  G_score_0_100  E_weighted  S_weighted  \
0   2002         0.2131         0.0489         0.1394    0.080391    0.017400   
1   2003         0.0936         0.3200         0.3286    0.035310    0.113868   
2   2004         0.0414         0.2690         0.2343    0.015618    0.095720   
3   2005         0.1212         0.9200         0.3587    0.045722    0.327371   
4   2006         0.9229         0.1174         0.6568    0.348160    0.041775   
5   2007         0.0924         0.3754         0.0356    0.034857    0.133582   
6   2008         0.2040         0.7920         0.4807    0.076958    0.281824   
7   2009         0.9496         0.0600         0.4942    0.358232    0.021350   
8   2010         0.3865         0.6850         0.1802    0.145805    0.243749   
9   2011         0.7821         0.5066         0.33

# XUẤT FILE

In [ ]:
import pandas as pd

# Gom tất cả kết quả vào 1 dictionary
dfs = {
    "china": china,
    "india": india,
    "japan": japan,
    "vietnam": vietnam,
    "singapore": singapore,
    "korea": korea
}

# Các cột cần xuất
cols = ["Year",
    "E_score_0_100", "S_score_0_100", "G_score_0_100",
    "E_weighted", "S_weighted", "G_weighted",
    "ESG_sub_entropy", "UI_score_0_100", "ESGUI_new"]

# Đường dẫn lưu file Excel trên Google Drive
output_path = "/content/drive/MyDrive/PETTIES/VERSION 1 - MANUSCRIPT/DATA/ESGUI_entropy_.xlsx"

# Xuất mỗi DataFrame thành 1 sheet
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for name, df in dfs.items():
        # Kiểm tra đủ cột trước khi xuất
        if set(cols).issubset(df.columns):
            df[cols].to_excel(writer, sheet_name=name[:31], index=False)
        else:
            print(f"⚠️ Sheet {name} thiếu cột — bỏ qua.")

print("✅ Đã xuất toàn bộ kết quả vào:")
print(output_path)